# 🚀 NGLab Getting Started Notebook

This polyglot notebook walks you through the NGLab codebase - a Multimodal Deep Reinforcement Learning platform for financial trading.

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                     TypeScript/React UI                        │
│                   (Tauri Desktop App)                          │
├─────────────────────────────────────────────────────────────────┤
│                      Tauri Commands                            │
├──────────────────────────┬──────────────────────────────────────┤
│     Rust Core Engine     │        Python ML Layer              │
│   • Order Book (CLOB)    │     • PyTorch Models                │
│   • Arena Simulation     │     • TorchRL Policies              │
│   • WebSocket Streaming  │     • Gymnasium Envs                │
├──────────────────────────┴──────────────────────────────────────┤
│                    PyO3 Bindings                               │
└─────────────────────────────────────────────────────────────────┘
```

## 1️⃣ Environment Setup

First, let's verify your environment is correctly configured.

In [ ]:
# Check Python version and key dependencies
import sys
print(f"Python: {sys.version}")

# Check PyTorch
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

# Check TorchRL
import torchrl
print(f"TorchRL: {torchrl.__version__}")

# Check Gymnasium
import gymnasium
print(f"Gymnasium: {gymnasium.__version__}")

In [ ]:
# Check NGLab Rust bindings
try:
    import nglab
    print("✅ NGLab Rust bindings loaded successfully!")
    
    # List available classes
    print("\nAvailable classes:")
    for name in dir(nglab):
        if not name.startswith('_'):
            print(f"  • nglab.{name}")
except ImportError as e:
    print(f"❌ NGLab not built. Run: maturin develop")
    print(f"   Error: {e}")

## 2️⃣ The Trading Environment

NGLab provides a Gymnasium-compatible trading environment powered by a Rust backend.

In [ ]:
import nglab
import numpy as np

# Create a trading environment
# Parameters: initial_capital, transaction_cost, lookback, max_steps, render, data_path
env = nglab.TradingEnv(
    initial_capital=10_000.0,
    transaction_cost=0.001,  # 0.1% per trade
    lookback=10,
    max_steps=1000,
    render=False,
    data_path=None  # Uses synthetic data
)

print(f"Environment created!")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

In [ ]:
# Reset and step through the environment
obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Initial info: {info}")

# Take a few random actions
total_reward = 0
for step in range(5):
    action = env.action_space.sample()  # Random action: 0=Hold, 1=Buy, 2=Sell
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    
    action_name = ['Hold', 'Buy', 'Sell'][action]
    print(f"Step {step+1}: {action_name} → Reward: {reward:.4f}, Portfolio: ${info['portfolio_value']:.2f}")

print(f"\nTotal reward after 5 steps: {total_reward:.4f}")

## 3️⃣ The Order Book (CLOB)

At the heart of NGLab is a high-performance Central Limit Order Book implementation.

In [ ]:
# Create an order book
ob = nglab.OrderBook()

# The order book starts empty
print("Empty order book created")
print(f"Best bid: {ob.best_bid()}")
print(f"Best ask: {ob.best_ask()}")
print(f"Spread: {ob.spread()}")

In [ ]:
# Add some orders
# add_order(price: f64, quantity: f64, is_bid: bool) -> order_id

# Add bid orders (buyers)
ob.add_order(99.50, 100.0, True)   # Bid at $99.50 for 100 units
ob.add_order(99.25, 200.0, True)   # Bid at $99.25 for 200 units
ob.add_order(99.00, 150.0, True)   # Bid at $99.00 for 150 units

# Add ask orders (sellers)
ob.add_order(100.50, 100.0, False)  # Ask at $100.50 for 100 units
ob.add_order(100.75, 200.0, False)  # Ask at $100.75 for 200 units
ob.add_order(101.00, 150.0, False)  # Ask at $101.00 for 150 units

print("Order book populated!")
print(f"Best bid: ${ob.best_bid():.2f}")
print(f"Best ask: ${ob.best_ask():.2f}")
print(f"Spread: ${ob.spread():.2f}")
print(f"Mid price: ${ob.mid_price():.2f}")

## 4️⃣ Multi-Asset Trading

For portfolio-level trading, use the MultiAssetEnv.

In [ ]:
# Create a multi-asset environment
multi_env = nglab.MultiAssetEnv(
    num_assets=5,
    initial_capital=100_000.0,
    transaction_cost=0.001,
    lookback=20,
    max_steps=500
)

obs, info = multi_env.reset()
print(f"Multi-asset observation shape: {obs.shape}")
print(f"Number of assets: 5")
print(f"Action space: Continuous weights for each asset")

## 5️⃣ Using Python Models

NGLab includes 30+ neural network architectures. Here's how to use them.

In [ ]:
import sys
sys.path.insert(0, 'python/src')

from models import ModelFactory

# List available models
print("Available models:")
for name in ModelFactory.list_models():
    print(f"  • {name}")

In [ ]:
# Create an LSTM model for time series
from models.time_series import LSTMModel

model = LSTMModel(
    input_size=10,      # Features per timestep
    hidden_size=64,     # LSTM hidden dimension
    num_layers=2,       # Stacked LSTM layers
    output_size=3,      # Actions: Hold, Buy, Sell
    dropout=0.1
)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create a Mamba (State Space Model) for efficient sequence modeling
from models.time_series import TSMamba

mamba_model = TSMamba(
    d_model=64,
    n_layers=4,
    d_state=16,
    input_size=10,
    output_size=3
)

print(f"TSMamba parameters: {sum(p.numel() for p in mamba_model.parameters()):,}")

# Mamba has O(N) complexity vs O(N²) for Transformers
# This makes it ideal for high-frequency trading with long sequences

## 6️⃣ Training with TorchRL

Here's a minimal PPO training loop using TorchRL.

In [ ]:
from torchrl.envs import GymWrapper
from torchrl.modules import ProbabilisticActor, ValueOperator
from torchrl.objectives import ClipPPOLoss
from tensordict.nn import TensorDictModule
import torch.nn as nn

# Wrap our Rust environment for TorchRL
env = nglab.TradingEnv(10_000.0, 0.001, 10, 100, False, None)
torchrl_env = GymWrapper(env)

print(f"TorchRL environment ready!")
print(f"Observation keys: {torchrl_env.observation_spec.keys()}")
print(f"Action keys: {torchrl_env.action_spec.keys()}")

In [ ]:
# Quick sanity check - collect some data
from torchrl.collectors import SyncDataCollector
from torchrl.data import ReplayBuffer, LazyTensorStorage

# This would be the start of a full training loop
# For a complete example, see: python/examples/train_ppo.py

print("For full training examples, run:")
print("  uv run python python/src/main.py task=ppo")
print("  uv run python python/src/main.py task=sac")
print("  uv run python python/src/main.py task=vae")

## 7️⃣ CLI Commands

NGLab provides a powerful CLI for training and evaluation.

In [ ]:
%%bash
# Show available CLI commands
cd ~/Repositories/nglab
uv run python python/src/main.py --help 2>/dev/null || echo "Run from project root"

In [ ]:
%%bash
# Show available tasks
echo "Available training tasks:"
echo "  • task=ppo     - Proximal Policy Optimization"
echo "  • task=sac     - Soft Actor-Critic"
echo "  • task=vae     - Variational Autoencoder"
echo "  • task=diffusion - Diffusion models"
echo ""
echo "Example:"
echo "  uv run python python/src/main.py task=ppo training.epochs=100"

## 8️⃣ Running the Dashboard

To launch the full Tauri desktop application:

In [ ]:
%%bash
echo "To run the dashboard:"
echo ""
echo "  cd ~/Repositories/nglab"
echo "  just dev       # Runs both Rust backend and React frontend"
echo ""
echo "Or separately:"
echo "  npm run tauri dev   # From typescript/ directory"
echo ""
echo "The dashboard includes:"
echo "  • Real-time charting with lightweight-charts"
echo "  • Order book visualization"
echo "  • Portfolio analytics"
echo "  • Strategy builder"
echo "  • Backtesting interface"
echo "  • ML training dashboard"

## 9️⃣ Project Structure Quick Reference

```
nglab/
├── rust/                   # High-performance Rust core
│   └── src/
│       ├── simulation/     # Arena, OrderBook, TradingEnv
│       ├── execution/      # TWAP, VWAP, POV algorithms
│       ├── web/            # HTTP/WebSocket handlers
│       └── models/         # ONNX inference
│
├── python/                 # ML training layer
│   └── src/
│       ├── models/         # 30+ neural architectures
│       ├── envs/           # Gymnasium wrappers
│       ├── policies/       # RL policies
│       └── pipeline/       # Training loops
│
├── typescript/             # Tauri React dashboard
│   └── src/
│       ├── components/     # UI components
│       └── hooks/          # State management
│
└── docs/                   # Architecture Decision Records
```

## 🎯 Next Steps

1. **Explore Models**: Check out `python/src/models/` for all available architectures
2. **Run Training**: `uv run python python/src/main.py task=ppo`
3. **Launch Dashboard**: `just dev`
4. **Read ADRs**: `docs/` contains architecture decisions
5. **Check Roadmap**: `ROADMAP.md` for planned features

---

**Happy Trading! 📈**